In [ ]:



import os
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/kaggle/input/datasets/sabbir4724/training-data"   
VAL_DIR   = "/kaggle/input/datasets/sabbir4724/testing-data"   

NUM_PROTOTYPES = 4        # K prototypes (set close to your expected number of classes)
FEATURE_DIM = 128         # projection head output dim (paper default: 128)
HIDDEN_DIM = 2048         # projection head hidden dim

GLOBAL_CROPS = 2          # number of global (high-res) crops
LOCAL_CROPS = 4           # number of local (low-res) crops
GLOBAL_SIZE = 224
LOCAL_SIZE = 96

BATCH_SIZE = 32
NUM_EPOCHS = 5
BASE_LR = 0.6
FINAL_LR = 0.0006
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 1e-6

EPSILON = 0.05             # Sinkhorn-Knopp regularization
SINKHORN_ITERS = 3
TEMPERATURE = 0.1
CROPS_FOR_ASSIGN = [0, 1]  # which crops get Sinkhorn assignment (the global ones)

QUEUE_LENGTH = 0            # set >0 (multiple of BATCH_SIZE) to enable memory queue
USE_QUEUE_AFTER_EPOCH = 15

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)


# ============================================================
# MULTI-CROP DATA AUGMENTATION (as in the original SwAV paper)
# ============================================================
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

color_jitter = transforms.RandomApply(
    [transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8
)

def make_crop_transform(size, scale):
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=scale),
        transforms.RandomHorizontalFlip(),
        color_jitter,
        transforms.RandomGrayscale(p=0.2),
        transforms.GaussianBlur(kernel_size=int(0.1 * size) // 2 * 2 + 1, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ])

global_transform = make_crop_transform(GLOBAL_SIZE, scale=(0.14, 1.0))
local_transform = make_crop_transform(LOCAL_SIZE, scale=(0.05, 0.14))


class MultiCropDataset(Dataset):
    """Wraps an ImageFolder-style dataset and returns GLOBAL_CROPS + LOCAL_CROPS
    differently-augmented views of each image, as in the SwAV paper."""

    def __init__(self, root):
        self.samples = datasets.ImageFolder(root).samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, _ = self.samples[idx]
        img = Image.open(path).convert("RGB")
        crops = [global_transform(img) for _ in range(GLOBAL_CROPS)]
        crops += [local_transform(img) for _ in range(LOCAL_CROPS)]
        return crops


def multicrop_collate(batch):
    n_crops = GLOBAL_CROPS + LOCAL_CROPS
    out = []
    for c in range(n_crops):
        out.append(torch.stack([sample[c] for sample in batch]))
    return out  # list of tensors, one per crop type: [B,3,H,W]


train_dataset = MultiCropDataset(TRAIN_DIR)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, drop_last=True,
                           collate_fn=multicrop_collate, persistent_workers=True,
                           prefetch_factor=2)

val_eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(GLOBAL_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_eval_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")
print(f"Val classes (for evaluation only): {val_dataset.classes}")



class SwAVModel(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM,
                 num_prototypes=NUM_PROTOTYPES):
        super().__init__()
        backbone = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # drop fc
        backbone_dim = backbone.fc.in_features

        self.projection_head = nn.Sequential(
            nn.Linear(backbone_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feature_dim),
        )
        self.prototypes = nn.Linear(feature_dim, num_prototypes, bias=False)

    def forward_backbone(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        return x

    def forward(self, crops):
        # crops: list of tensors of possibly different sizes -> group by size.
        # Crops arrive already contiguous by type (all globals, then all
        # locals), so we must preserve that order when grouping — sorting by
        # size value (e.g. via np.unique) would put the 96px locals before
        # the 224px globals and misalign the concatenated batches.
        sizes = torch.tensor([c.shape[-1] for c in crops])
        _, counts = torch.unique_consecutive(sizes, return_counts=True)
        idx_crops = torch.cumsum(counts, dim=0).tolist()

        start = 0
        embeddings = []
        for end in idx_crops:
            batch = torch.cat(crops[start:end], dim=0).to(DEVICE, non_blocking=True)
            feat = self.forward_backbone(batch)
            embeddings.append(feat)
            start = end
        embeddings = torch.cat(embeddings, dim=0)

        z = self.projection_head(embeddings)
        z = F.normalize(z, dim=1, p=2)
        p = self.prototypes(z)
        return z, p

    @torch.no_grad()
    def extract_features(self, x):
        feat = self.forward_backbone(x)
        z = self.projection_head(feat)
        z = F.normalize(z, dim=1, p=2)
        return z

    @torch.no_grad()
    def normalize_prototypes(self):
        w = self.prototypes.weight.data.clone()
        w = F.normalize(w, dim=1, p=2)
        self.prototypes.weight.copy_(w)


model = SwAVModel().to(DEVICE)


# ============================================================
# SINKHORN-KNOPP (online optimal transport for cluster assignment)
# ============================================================
@torch.no_grad()
def sinkhorn(scores, epsilon=EPSILON, n_iters=SINKHORN_ITERS):
    Q = torch.exp(scores / epsilon).t()  # [K, B]
    B = Q.shape[1]
    K = Q.shape[0]

    sum_Q = torch.sum(Q)
    Q /= sum_Q

    for _ in range(n_iters):
        sum_rows = torch.sum(Q, dim=1, keepdim=True)
        Q /= sum_rows
        Q /= K

        sum_cols = torch.sum(Q, dim=0, keepdim=True)
        Q /= sum_cols
        Q /= B

    Q *= B
    return Q.t()  # [B, K]


# ============================================================
# LR SCHEDULE (cosine with warmup, as in the paper)
# ============================================================
def build_lr_schedule(steps_per_epoch):
    total_steps = NUM_EPOCHS * steps_per_epoch
    warmup_steps = WARMUP_EPOCHS * steps_per_epoch
    schedule = []
    for step in range(total_steps):
        if step < warmup_steps:
            lr = BASE_LR * step / max(1, warmup_steps)
        else:
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            lr = FINAL_LR + 0.5 * (BASE_LR - FINAL_LR) * (1 + math.cos(math.pi * progress))
        schedule.append(lr)
    return schedule


# ============================================================
# QUEUE (optional memory bank of past embeddings, as in the paper)
# ============================================================
queue = None
if QUEUE_LENGTH > 0:
    queue = torch.zeros(len(CROPS_FOR_ASSIGN), QUEUE_LENGTH, FEATURE_DIM).to(DEVICE)


# ============================================================
# TRAINING (SwAV swapped-prediction loss)
# ============================================================
def train_one_epoch(net, loader, optimizer, lr_schedule, step_offset, epoch):
    net.train()
    total_loss, total_n = 0.0, 0
    global queue

    pbar = tqdm(enumerate(loader), total=len(loader),
                desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", leave=True)

    for it, crops in pbar:
        step = step_offset + it
        lr = lr_schedule[min(step, len(lr_schedule) - 1)]
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        net.normalize_prototypes()

        bs = crops[0].shape[0]
        embeddings, output = net(crops)

        loss = 0
        for i, crop_id in enumerate(CROPS_FOR_ASSIGN):
            with torch.no_grad():
                out = output[bs * crop_id: bs * (crop_id + 1)].detach()

                if queue is not None and epoch >= USE_QUEUE_AFTER_EPOCH:
                    out = torch.cat((torch.mm(queue[i], net.prototypes.weight.t()), out))
                    queue[i, bs:] = queue[i, :-bs].clone()
                    queue[i, :bs] = embeddings[bs * crop_id: bs * (crop_id + 1)].detach()

                q = sinkhorn(out)[-bs:]

            subloss = 0
            for v in range(len(crops)):
                if v == crop_id:
                    continue
                p = F.softmax(output[bs * v: bs * (v + 1)] / TEMPERATURE, dim=1)
                subloss -= torch.mean(torch.sum(q * torch.log(p + 1e-12), dim=1))
            loss += subloss / (len(crops) - 1)

        loss /= len(CROPS_FOR_ASSIGN)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * bs
        total_n += bs

        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.5f}")

    return total_loss / total_n


# ============================================================
# EVALUATION on the separate validation dataset
# ============================================================
@torch.no_grad()
def extract_val_features(net):
    net.eval()
    feats = []
    for images, _ in val_loader:
        images = images.to(DEVICE)
        z = net.extract_features(images)
        feats.append(z.cpu().numpy())
    return np.concatenate(feats, axis=0)


def hungarian_match(pred_clusters, true_labels, num_clusters, num_classes):
    size = max(num_clusters, num_classes)
    cm = np.zeros((size, size), dtype=np.int64)
    for p, t in zip(pred_clusters, true_labels):
        cm[p, t] += 1
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {r: c for r, c in zip(row_ind, col_ind)}
    return np.array([mapping[p] for p in pred_clusters])


def evaluate_on_val(net):
    val_features = extract_val_features(net)
    val_true_labels = np.array(val_dataset.targets)
    num_classes = len(val_dataset.classes)

    km = KMeans(n_clusters=NUM_PROTOTYPES, n_init=20, random_state=SEED)
    val_clusters = km.fit_predict(val_features)

    mapped_preds = hungarian_match(val_clusters, val_true_labels, NUM_PROTOTYPES, num_classes)

    acc = accuracy_score(val_true_labels, mapped_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_true_labels, mapped_preds, average="macro", zero_division=0
    )
    return acc, precision, recall, f1


# ============================================================
# MAIN LOOP
# ============================================================
def main():
    optimizer = optim.SGD(model.parameters(), lr=BASE_LR, momentum=0.9,
                           weight_decay=WEIGHT_DECAY)
    steps_per_epoch = len(train_loader)
    lr_schedule = build_lr_schedule(steps_per_epoch)
    print(f"Steps per epoch: {steps_per_epoch} (batch_size={BATCH_SIZE})")

    for epoch in range(NUM_EPOCHS):
        t0 = time.time()
        step_offset = epoch * steps_per_epoch
        train_loss = train_one_epoch(model, train_loader, optimizer,
                                      lr_schedule, step_offset, epoch)
        dt = time.time() - t0
        print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] loss={train_loss:.4f} time={dt:.1f}s")

    acc, precision, recall, f1 = evaluate_on_val(model)
    print("\n=== Validation Results (k-means clusters, Hungarian-matched vs true labels) ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    torch.save(model.state_dict(), "swav_model.pth")
    print("\nModel saved to swav_model.pth")


if __name__ == "__main__":
    main()

In [ ]:


import time
import torch
import numpy as np
from torchvision import models
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# CONFIG
# ============================================================
CHECKPOINT_PATH = "swav_model.pth"   # <-- path to your trained model
IMAGE_SIZE = 224
FEATURE_DIM = 128
HIDDEN_DIM = 2048
NUM_PROTOTYPES = 4          # must match the value used during training

NUM_WARMUP = 20
NUM_RUNS = 200               # number of timed single-image forward passes

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# MODEL (same architecture as training script)
# ============================================================
class SwAVModel(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM,
                 num_prototypes=NUM_PROTOTYPES):
        super().__init__()
        backbone = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        backbone_dim = backbone.fc.in_features

        self.projection_head = nn.Sequential(
            nn.Linear(backbone_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feature_dim),
        )
        self.prototypes = nn.Linear(feature_dim, num_prototypes, bias=False)

    def forward(self, x):
        feat = self.backbone(x)
        feat = torch.flatten(feat, 1)
        z = self.projection_head(feat)
        z = F.normalize(z, dim=1, p=2)
        p = self.prototypes(z)
        return p


# ============================================================
# LOAD MODEL
# ============================================================
model = SwAVModel().to(DEVICE)
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

# BatchNorm needs batch_size > 1 in train mode, but eval mode uses running
# stats, so batch_size=1 inference works fine here.


# ============================================================
# PARAMS (M)
# ============================================================
total_params = sum(p.numel() for p in model.parameters())
params_m = total_params / 1e6


# ============================================================
# TIME / IMAGE (ms) + FPS
# ============================================================
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)

with torch.no_grad():
    # Warm-up (important for stable GPU timing — excludes CUDA context /
    # cuDNN autotune overhead from the measurement)
    for _ in range(NUM_WARMUP):
        _ = model(dummy_input)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    timings = []
    for _ in range(NUM_RUNS):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        _ = model(dummy_input)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        timings.append((t1 - t0) * 1000.0)  # ms

timings = np.array(timings)
mean_time_ms = timings.mean()
std_time_ms = timings.std()
fps = 1000.0 / mean_time_ms


# ============================================================
# REPORT
# ============================================================
print("=" * 50)
print("Model Complexity Benchmark")
print("=" * 50)
print(f"Device          : {DEVICE}")
print(f"Params (M)      : {params_m:.2f}")
print(f"Time/Image (ms) : {mean_time_ms:.2f} ± {std_time_ms:.2f}")
print(f"FPS             : {fps:.2f}")
print("=" * 50)